In [1]:
import torch
import numpy as np
import os
import re
from transformers import BertTokenizer, BertModel
from typing import Dict, List
import pickle

# 配置路径
MODEL_PATH = "bert-base-chinese-local"  # 本地BERT模型路径
RAG_DIR = "rag"  # TXT文件目录
VECTOR_DB_PATH = "vector_db.pkl"  # 向量库保存路径
CHUNK_SIZE = 512  # 分块大小（字符数）
OVERLAP_SIZE = 50  # 块之间的重叠字符数

# 初始化本地BERT模型和分词器
print("加载本地BERT模型...")
tokenizer = BertTokenizer.from_pretrained(MODEL_PATH)
model = BertModel.from_pretrained(MODEL_PATH)
model.eval()  # 设置为评估模式


def read_txt_files(txt_dir: str) -> Dict[str, str]:
    """读取所有TXT文件，返回字典：{文件名: 内容}"""
    txt_contents = {}
    for filename in os.listdir(txt_dir):
        if filename.endswith(".txt"):
            filepath = os.path.join(txt_dir, filename)
            with open(filepath, 'r', encoding='utf-8') as f:
                content = f.read().strip()
                txt_contents[filename] = content
    print(f"共读取 {len(txt_contents)} 个TXT文件")
    return txt_contents


def split_text_into_chunks(text: str, chunk_size: int = CHUNK_SIZE, overlap_size: int = OVERLAP_SIZE) -> List[str]:
    """
    将文本分割成固定大小的块
    保持句子完整性，避免在句子中间切分
    """
    # 按句子分割（中文标点）
    sentences = re.split(r'([。！？；\.\?!;])', text)

    # 重新组合句子，保持标点
    sentences_with_punct = []
    for i in range(0, len(sentences) - 1, 2):
        if i + 1 < len(sentences):
            sentences_with_punct.append(sentences[i] + sentences[i + 1])
        else:
            sentences_with_punct.append(sentences[i])

    if len(sentences) % 2 == 1:  # 处理最后一个单独的句子
        sentences_with_punct.append(sentences[-1])

    # 按句子合并成块
    chunks = []
    current_chunk = ""

    for sentence in sentences_with_punct:
        # 如果当前句子加上当前块超过chunk_size，则保存当前块
        if len(current_chunk) + len(sentence) > chunk_size and current_chunk:
            chunks.append(current_chunk.strip())
            # 保留重叠部分
            overlap_start = max(0, len(current_chunk) - overlap_size)
            current_chunk = current_chunk[overlap_start:] + sentence
        else:
            current_chunk += sentence

    # 添加最后一个块
    if current_chunk:
        chunks.append(current_chunk.strip())

    return chunks


def get_bert_embedding(text: str, max_length: int = 512) -> np.ndarray:
    """使用BERT获取文本的嵌入向量（取CLS token的表示）"""
    # 分词和编码
    inputs = tokenizer(
        text,
        max_length=max_length,
        padding='max_length',
        truncation=True,
        return_tensors="pt"
    )

    # 推理
    with torch.no_grad():
        outputs = model(**inputs)
        # 取CLS token的表示（最后一层隐藏状态）
        cls_embedding = outputs.last_hidden_state[:, 0, :].squeeze().numpy()

    return cls_embedding


def process_document_chunks(filename: str, content: str) -> List[Dict]:
    """处理单个文档，将其分块并生成向量"""
    # 分块
    chunks = split_text_into_chunks(content, CHUNK_SIZE, OVERLAP_SIZE)
    print(f"  '{filename}' 分割为 {len(chunks)} 个块")

    # 为每个块生成向量
    chunk_data = []
    for i, chunk in enumerate(chunks):
        # 生成向量
        embedding = get_bert_embedding(chunk)

        # 存储块信息
        chunk_info = {
            "chunk_id": i,
            "chunk_text": chunk,
            "embedding": embedding,
            "embedding_dim": embedding.shape[-1],
            "chunk_length": len(chunk),
            "parent_doc": filename,
            "chunk_range": f"{i + 1}/{len(chunks)}"
        }
        chunk_data.append(chunk_info)

    return chunk_data


def build_vector_db(txt_contents: Dict[str, str]) -> Dict[str, dict]:
    """构建向量数据库，返回字典结构"""
    vector_db = {}
    total_chunks = 0

    for idx, (filename, content) in enumerate(txt_contents.items()):
        print(f"处理文件 [{idx + 1}/{len(txt_contents)}]: {filename}")

        # 处理文档分块
        chunks_data = process_document_chunks(filename, content)
        total_chunks += len(chunks_data)

        # 存储到向量库
        vector_db[filename] = {
            "filename": filename,
            "original_content": content,
            "total_chunks": len(chunks_data),
            "chunks": chunks_data,
            "metadata": {
                "original_length": len(content),
                "avg_chunk_length": sum(c["chunk_length"] for c in chunks_data) / len(chunks_data) if chunks_data else 0
            }
        }

    print(f"\n总计处理 {len(vector_db)} 个文档，{total_chunks} 个文本块")
    return vector_db


def save_vector_db(vector_db: Dict[str, dict], save_path: str):
    """保存向量数据库到文件"""
    with open(save_path, 'wb') as f:
        pickle.dump(vector_db, f)
    print(f"向量库已保存到: {save_path}")


if __name__ == "__main__":
    # 1. 读取TXT文件
    print("步骤1: 读取TXT文件...")
    txt_contents = read_txt_files(RAG_DIR)

    # 2. 构建向量库（带分块）
    print("\n步骤2: 构建向量库（分块处理）...")
    print(f"分块参数: chunk_size={CHUNK_SIZE}, overlap={OVERLAP_SIZE}")
    vector_db = build_vector_db(txt_contents)

    # 3. 保存向量库
    print("\n步骤3: 保存向量库...")
    save_vector_db(vector_db, VECTOR_DB_PATH)

    # 4. 输出统计信息
    print("\n" + "=" * 60)
    print("向量库构建完成！")
    print(f"文档数量: {len(vector_db)}")

    # 计算总文本块数
    total_chunks = sum(doc["total_chunks"] for doc in vector_db.values())
    print(f"文本块总数: {total_chunks}")

Some weights of BertModel were not initialized from the model checkpoint at bert-base-chinese-local and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


加载本地BERT模型...
步骤1: 读取TXT文件...
共读取 20 个TXT文件

步骤2: 构建向量库（分块处理）...
分块参数: chunk_size=512, overlap=50
处理文件 [1/20]: 1944383.txt
  '1944383.txt' 分割为 22 个块
处理文件 [2/20]: 1944399.txt
  '1944399.txt' 分割为 20 个块
处理文件 [3/20]: 1944404.txt
  '1944404.txt' 分割为 21 个块
处理文件 [4/20]: 1944407.txt
  '1944407.txt' 分割为 35 个块
处理文件 [5/20]: 1944422.txt
  '1944422.txt' 分割为 6 个块
处理文件 [6/20]: 1944435.txt
  '1944435.txt' 分割为 23 个块
处理文件 [7/20]: 1944441.txt
  '1944441.txt' 分割为 33 个块
处理文件 [8/20]: 1944466.txt
  '1944466.txt' 分割为 26 个块
处理文件 [9/20]: 1944470.txt
  '1944470.txt' 分割为 30 个块
处理文件 [10/20]: 1961211.txt
  '1961211.txt' 分割为 11 个块
处理文件 [11/20]: 1961213.txt
  '1961213.txt' 分割为 10 个块
处理文件 [12/20]: 1961219.txt
  '1961219.txt' 分割为 9 个块
处理文件 [13/20]: 1961230.txt
  '1961230.txt' 分割为 7 个块
处理文件 [14/20]: 1961232.txt
  '1961232.txt' 分割为 6 个块
处理文件 [15/20]: 1961236.txt
  '1961236.txt' 分割为 13 个块
处理文件 [16/20]: 1961238.txt
  '1961238.txt' 分割为 10 个块
处理文件 [17/20]: 1961239.txt
  '1961239.txt' 分割为 13 个块
处理文件 [18/20]: 1961242.txt
  '19

2.基于代码一生成的向量库实现RAG
以下代码完成了RAG流程：初始化系统 → 用户提问 → 向量检索 → 构建提示词 → LLM生成答案。替换其中的API_key，运行代码，输出最终问题的答案。

API_key获取：进入 https://bailian.console.aliyun.com/?spm=5176.29597918.nav-v2-dropdown-menu-0.d_main_2_0_1.52007b08KzGnZU&tab=model&scm=20140722.M_10904465._.V_1#/model-market ，完成登入后，点击左下角“密钥管理”，点击“创建API KEY”，获得密钥。

In [2]:
import torch
import numpy as np
import pickle
from transformers import BertTokenizer, BertModel
from openai import OpenAI
from typing import List, Dict
import time


def cosine_similarity(vec1: np.ndarray, vec2: np.ndarray) -> float:
    """计算余弦相似度"""
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2) + 1e-8)


class ChunkBasedRAGSystem:
    def __init__(self, vector_db_path: str, model_path: str = "bert-base-chinese-local"):
        """初始化RAG系统（基于分块的向量库）"""
        print("=" * 60)
        print("初始化分块RAG问答系统")
        print("=" * 60)

        # 加载向量库
        print("1. 加载向量库...")
        start_time = time.time()
        
        # 直接加载原始向量库
        with open(vector_db_path, 'rb') as f:
            self.vector_db = pickle.load(f)
        
        print(f"   加载完成，共 {len(self.vector_db)} 个文档，耗时: {time.time() - start_time:.2f}秒")

        # 加载BERT模型用于查询向量化
        print("2. 加载BERT模型...")
        start_time = time.time()
        self.tokenizer = BertTokenizer.from_pretrained(model_path)
        self.model = BertModel.from_pretrained(model_path)
        self.model.eval()
        print(f"   模型加载完成，耗时: {time.time() - start_time:.2f}秒")

        # 初始化OpenAI客户端
        print("3. 初始化API客户端...")
        self.client = OpenAI(
            api_key="sk-07ba5bd13252402d94aad91eb52473cb",#替换为你的API_key,
            base_url="https://dashscope.aliyuncs.com/compatible-mode/v1"
        )

        # 准备向量数据
        print("4. 准备向量数据...")
        start_time = time.time()
        self.chunk_embeddings = []
        self.chunks = []

        for doc_data in self.vector_db.values():
            for chunk in doc_data["chunks"]:
                self.chunk_embeddings.append(chunk["embedding"])
                self.chunks.append({
                    "doc_filename": doc_data["filename"],
                    "chunk_id": chunk["chunk_id"],
                    "chunk_text": chunk["chunk_text"],
                    "parent_doc": chunk["parent_doc"],
                    "chunk_range": chunk["chunk_range"]
                })

        self.chunk_embeddings = np.array(self.chunk_embeddings)
        total_chunks = len(self.chunks)
        print(f"   准备完成，共 {total_chunks} 个文本块，耗时: {time.time() - start_time:.2f}秒")

        print(f"系统初始化完成！\n")

    def vectorize_query(self, query: str) -> np.ndarray:
        """向量化查询文本"""
        inputs = self.tokenizer(
            query,
            max_length=512,
            padding='max_length',
            truncation=True,
            return_tensors="pt"
        )

        with torch.no_grad():
            outputs = self.model(**inputs)
            query_embedding = outputs.last_hidden_state[:, 0, :].squeeze().numpy()

        return query_embedding

    def retrieve_with_embedding(self, query: str, top_k: int = 5) -> List[Dict]:
        """使用向量相似度检索最相关的文本块"""
        query_vector = self.vectorize_query(query)

        # 计算余弦相似度
        similarities = []
        for i, chunk_vector in enumerate(self.chunk_embeddings):
            sim = cosine_similarity(query_vector, chunk_vector)
            similarities.append((sim, i))

        # 排序并获取top-k
        similarities.sort(key=lambda x: x[0], reverse=True)

        relevant_chunks = []
        seen_docs = set()  # 用于去重，避免同一文档的多个块

        for sim, idx in similarities[:top_k * 2]:  # 获取更多结果以便去重
            if len(relevant_chunks) >= top_k:
                break

            chunk_info = self.chunks[idx]
            doc_name = chunk_info["doc_filename"]

            # 避免同一文档的多个块
            if doc_name in seen_docs and len(relevant_chunks) > 0:
                continue

            seen_docs.add(doc_name)

            relevant_chunks.append({
                "doc_filename": chunk_info["doc_filename"],
                "chunk_id": chunk_info["chunk_id"],
                "chunk_text": chunk_info["chunk_text"],
                "similarity": float(sim)
            })

        return relevant_chunks

    def build_prompt(self, question: str, relevant_chunks: List[Dict]) -> str:
        """构建高质量的提示词（基于文本块）"""

        # 构建上下文
        context_parts = []
        for i, chunk in enumerate(relevant_chunks, 1):
            # 使用文本块内容
            chunk_text = chunk["chunk_text"]
            context_parts.append(chunk_text)

        context = "\n\n".join(context_parts)

        # 构建提示词
        prompt = f"""你是一个专业的农业科学助手，请基于以下提供的文档内容，准确回答用户的问题。

提供的相关文档内容：
{context}

用户的问题：{question}

请根据以上文档内容，提供专业、准确、完整的回答。

现在，请开始回答："""

        return prompt

    def answer_question(self, question: str, question_num: int = None) -> str:
        """回答单个问题，返回答案"""
        if question_num:
            print(f"\n{'=' * 80}")
            print(f"问题 {question_num}: {question}")
            print('=' * 80)

        # 检索相关文本块
        relevant_chunks = self.retrieve_with_embedding(question, top_k=3)

        if not relevant_chunks:
            return "抱歉，在知识库中没有找到相关信息。"

        # 构建提示词
        prompt = self.build_prompt(question, relevant_chunks)

        # 调用LLM
        try:
            completion = self.client.chat.completions.create(
                model="deepseek-v3",
                messages=[{"role": "user", "content": prompt}],
                max_tokens=1000,
                temperature=0.1
            )
            answer = completion.choices[0].message.content
        except Exception as e:
            answer = f"调用API时出错：{str(e)}"

        return answer


def main():
    """主函数：处理10个问题"""

    # 定义10个问题
    questions = [
        "胡柚果实采后枯水发生的根本原因是什么？有哪些关键酶活性变化与其相关？",
        "套袋对提高惠民短枝红富士苹果品质有哪些具体效应？",
        "桃果实发育过程中褐变与哪些因素相关？多酚氧化酶（PPO）的热稳定性如何？",
        "柑桔体细胞杂种在抗性方面有哪些优势？请举例说明。",
        "蕉柑的起源和分类地位如何？有哪些证据支持？",
        "低温胁迫下，香蕉与大蕉在SOD活性和ABA含量上有何差异？多效唑如何处理这些差异？",
        "石榴和桃在NaCl胁迫下对Na⁺和K⁺的吸收与转运有何不同？",
        "沙田柚自交不亲和的表现机制是什么？属于哪种不亲和类型？",
        "我国果树营养研究在哪些方面取得了进展？尚存哪些问题？",
        "桃树根癌病的生物防治方法有哪些？K84菌液的防治效果如何？"
    ]

    print("开始分块RAG问答系统处理流程")
    print(f"待处理问题数: {len(questions)}")
    print()

    # 1. 初始化RAG系统
    rag_system = ChunkBasedRAGSystem("vector_db.pkl")

    # 2. 处理每个问题并打印答案
    all_answers = []
    total_start_time = time.time()

    for i, question in enumerate(questions, 1):
        # 获取答案
        answer = rag_system.answer_question(question, question_num=i)
        all_answers.append(answer)
        
        # 打印答案
        print(f"\n答案 {i}:")
        print("-" * 80)
        print(answer)
        print("-" * 80)

    total_time = time.time() - total_start_time

    # 3. 打印统计信息
    print(f"\n{'=' * 80}")
    print("所有问题处理完成！")
    print(f"总耗时: {total_time:.2f}秒")
    print(f"平均每个问题: {total_time / len(questions):.2f}秒")
    print()


if __name__ == "__main__":
    main()

Some weights of BertModel were not initialized from the model checkpoint at bert-base-chinese-local and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


开始分块RAG问答系统处理流程
待处理问题数: 10

初始化分块RAG问答系统
1. 加载向量库...
   加载完成，共 20 个文档，耗时: 0.00秒
2. 加载BERT模型...
   模型加载完成，耗时: 0.12秒
3. 初始化API客户端...
4. 准备向量数据...
   准备完成，共 328 个文本块，耗时: 0.00秒
系统初始化完成！


问题 1: 胡柚果实采后枯水发生的根本原因是什么？有哪些关键酶活性变化与其相关？

答案 1:
--------------------------------------------------------------------------------
根据提供的文档内容，胡柚果实采后枯水的根本原因与**果皮和果肉组织中抗氧化酶活性变化**密切相关，具体表现为**SOD（超氧化物歧化酶）和POD（过氧化物酶）活性的相对变化**。这些酶活性的异常可能导致活性氧代谢失衡，进而引发细胞膜脂过氧化和细胞结构破坏，最终促使果实组织失水、质地劣变。

### 关键酶活性变化：
1. **SOD活性**  
   SOD是清除超氧自由基的关键酶，其活性下降可能导致自由基积累，加剧膜脂过氧化，破坏细胞膜完整性，促进枯水发生。

2. **POD活性**  
   POD参与清除过氧化物和调节氧化应激。文档指出枯水过程中POD活性发生显著变化，其失调可能加速细胞降解和水分流失。

### 其他潜在关联因素（间接提示）：
- **膜脂脂肪酸组成**（虽未直接提及胡柚，但柑橘类研究中IUFA与抗逆性相关）  
  若枯水伴随膜脂稳定性下降（如亚麻酸/棕榈酸比值降低），可能进一步加剧细胞膜透性增加，导致水分流失。

### 结论：
胡柚枯水的根本机制需结合**氧化应激损伤**和**细胞膜降解**两方面分析，而**SOD与POD活性的动态变化**是直接关联的关键生化指标。建议进一步通过实验验证这些酶在枯水不同阶段的具体作用。
--------------------------------------------------------------------------------

问题 2: 套袋对提高惠民短枝红富士苹果品质有哪些具体效应？

答案 2:
---------------------------------